In [7]:
import pretty_midi
import json
import numpy as np
from pathlib import Path


DATA_DIR = Path("../../data/dataset/midi")
data_dir = sorted(DATA_DIR.glob("*.mid"))
list(DATA_DIR.glob("*.mid"))[:3]

[PosixPath('../../data/dataset/midi/Iron Maiden - Purgatory.mid'),
 PosixPath('../../data/dataset/midi/Bluegrass - Red-Haired Boy.mid'),
 PosixPath('../../data/dataset/midi/DepecheMode_NewLife.1.mid')]

In [8]:
def quick_bpm(pm: pretty_midi.PrettyMIDI) -> float:
    """
    Estimate the global tempo (BPM) of a MIDI file from its embedded
    tempo changes.

    This function retrieves the tempo values stored in the MIDI object
    via `pretty_midi.PrettyMIDI.get_tempo_changes()` and returns the
    median tempo as a single BPM value. If no tempo information is
    available, it falls back to a default value of 120 BPM.

    Parameters
    ----------
    pm : pretty_midi.PrettyMIDI
        Parsed MIDI object from which to extract tempo information.

    Returns
    -------
    float
        Estimated global tempo in beats per minute (BPM).
    """
    tempos = pm.get_tempo_changes()[1]
    if len(tempos) == 0:
        return 120.0
    return float(np.median(tempos))

In [9]:
def quick_role(inst: pretty_midi.Instrument) -> str:
    """
    Assign a coarse-grained role label to a MIDI instrument track.

    The role is determined using simple heuristics:
      - If `inst.is_drum` is True, the role is "drums".
      - If the track has no notes, the role is "empty".
      - Otherwise, the role is inferred from the pitch range:
        tracks with pitches roughly in the bass register
        (MIDI note range ~[52, 80]) are labeled "bass",
        and all others are labeled "harmonic".

    Parameters
    ----------
    inst : pretty_midi.Instrument
        MIDI instrument track to classify.

    Returns
    -------
    str
        One of {"drums", "bass", "harmonic", "empty"} indicating the
        inferred role of the track.
    """
    if inst.is_drum:
        return "drums"
    if not inst.notes:
        return "empty"
    pitches = [n.pitch for n in inst.notes]
    lo, hi = min(pitches), max(pitches)
    if lo <= 52 and hi <= 80:
        return "bass"
    return "harmonic"

In [10]:
def summarize_tracks(pm: pretty_midi.PrettyMIDI) -> list:
    tracks = []
    for i, inst in enumerate(pm.instruments):
        if not inst.notes:
            continue
        pitches = [n.pitch for n in inst.notes]
        if not inst.notes: continue
        lo, hi = min(pitches), max(pitches)
        role = quick_role(inst)
        tracks.append({
            "index": i,
            "role": role,
            "is_drum": bool(inst.is_drum),
            "program": int(inst.program),
            "pitch_lo": int(lo),
            "pitch_hi": int(hi),
            "n_notes": len(pitches),
            "name": inst.name
        })
    return tracks

In [11]:
def choose_main_tracks(tracks):
    drum_candidates = [t for t in tracks if t["role"] == "drums"]
    bass_candidates = [t for t in tracks if t["role"] == "bass"]
    harm_candidates = [t for t in tracks if t["role"] == "harmonic"]

    drum_main = max(drum_candidates, key=lambda t: t["n_notes"]) if drum_candidates else None
    bass_main = max(bass_candidates, key=lambda t: t["n_notes"]) if bass_candidates else None
    harm_main = max(harm_candidates, key=lambda t: t["n_notes"]) if harm_candidates else None

    return drum_main, bass_main, harm_main

In [12]:
def build_manifest(midi_paths, out_path: Path):
    n_total         = 0
    n_ok            = 0
    n_missing_roles = 0

    with out_path.open("w", encoding="utf-8") as f_out:
        for p in midi_paths:
            n_total += 1
            try:
                pm = pretty_midi.PrettyMIDI(str(p))
            
            except:
                continue

            bpm = quick_bpm(pm)
            tracks = summarize_tracks(pm)
            d, b, h = choose_main_tracks(tracks)

            if d is None or b is None or h is None:
                n_missing_roles += 1
                continue

            row = {
                "song_id": p.stem,
                "midi_path": str(p),
                "bpm": bpm,
                "drum_idx": d["index"],
                "bass_idx": b["index"],
                "harm_idx": h["index"]
            }
            f_out.write(json.dumps(row) + "\n")
            n_ok += 1

    print("Total        :", n_total)
    print("Missing roles:", n_missing_roles)
    print("Usable       :", n_ok)
    print("Manifest     :", out_path)
    
manifest_path = Path("../../data/json/manifest.jsonl")

build_manifest(data_dir, manifest_path)

/opt/anaconda3/lib/python3.12/site-packages/pretty_midi/pretty_midi.py:122: RuntimeWarning: Tempo, Key or Time signature change events found on non-zero tracks.  This is not a valid type 0 or type 1 MIDI file.  Tempo, Key or Time Signature may be wrong.
  warnings.warn(


Total        : 8168
Missing roles: 1325
Usable       : 6843
Manifest     : ../../data/json/manifest.jsonl
